# ADVC revision W1 checks (branch `revision-w1`, TinyImageNet, Kaggle T4)

**Before running (right sidebar):**
1. Settings -> Accelerator -> **GPU T4 x2** (one GPU is used)
2. Settings -> Internet -> **On**
3. Add Input -> **jainmanav20/advc-results** (checkpoints) and **albert1994/tinyimagenet** (images)

Then **Run All**. Read-only for your inputs; everything is written to `/kaggle/working/w1/` (Output tab).

| Cell | What | Estimate (unmeasured) | Default |
|---|---|---|---|
| 1-4 | clone branch, install pins, link checkpoints, build tiny_if | ~5-8 min (mostly CPU data prep) | on |
| 5 | env report | <1 min | on |
| 6 | unit tests (CPU + GPU bias-guard test) | ~1-3 min | on |
| 7 | smoke: load/forward/backward/FGSM/PGD-20 on 1 batch, fp32/int8/int4 | ~1-3 min GPU | on |
| 8 | INT8 checkpoint reload probe (A-13 hypothesis 1) | ~1 min | on |
| 9 | July INT4 pickle inspection (CPU only) | ~1 min | on |
| 10 | calibrate (timing + TF32 on/off) | ~3-5 min GPU | **off**: set `RUN_CALIBRATE = True` |
| 11 | repro check, 1k images, clean/FGSM/PGD-20 x 3 precisions | ~8-12 min GPU | **off**: set `RUN_REPRO = True` |
| 12 | summary: copy its output back to Claude | <1 min | on |

Default run (cells 10-11 off): roughly 10-15 min total, of which ~3-5 min is GPU. All times are estimates, not measurements.

In [ ]:
# 0. Switches (cells 10 and 11 cost more than 5 min of GPU together; turn them on only if you approve)
RUN_CALIBRATE = False
RUN_REPRO = False
BRANCH = 'revision-w1'
W1 = '/kaggle/working/w1'
OUT_ROOT = W1 + '/results'

In [ ]:
# 1. GPU check + clone the revision branch + helper to run and log commands
import os, subprocess, sys, time
os.makedirs(W1, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
print('branch:', subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert 'name: "tiny-imagenet"' in open('configs/base.yaml').read(), 'configs/base.yaml dataset.name must be tiny-imagenet'

TIMES = {}
def sh(name, cmd, env=None):
    """Run cmd (list), stream output, save it to W1/<name>.log, record wall time and exit code."""
    t0 = time.time()
    log_path = f'{W1}/{name}.log'
    with open(log_path, 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env=dict(os.environ, **(env or {})))
        for line in p.stdout:
            log.write(line)
            print(line, end='')
        p.wait()
    TIMES[name] = {'seconds': round(time.time() - t0, 1), 'exit': p.returncode}
    print(f'\n[{name}] exit={p.returncode} wall={TIMES[name]["seconds"]}s  log={log_path}')
    return p.returncode

In [ ]:
# 2. Libraries: bitsandbytes pinned to 0.50.2 (requirements.lock); torch/timm are Kaggle's (recorded by env_report)
!pip install -q --no-deps bitsandbytes==0.50.2   # --no-deps: never let pip replace Kaggle's torch
!pip install -q torchattacks pytest
import torch, timm, torchattacks, bitsandbytes
print('torch', torch.__version__, '| timm', timm.__version__, '| torchattacks', torchattacks.__version__, '| bitsandbytes', bitsandbytes.__version__)
assert bitsandbytes.__version__ == '0.50.2', 'bitsandbytes pin did not take effect: restart the kernel and rerun from cell 0'

In [ ]:
# 3. Link the TinyImageNet checkpoints from advc-results (read-only) to where the code expects them
import glob
hits = glob.glob('/kaggle/input/**/at_int8_epoch01.pt', recursive=True)
print('found:', hits)
assert hits, 'No checkpoints found: attach jainmanav20/advc-results'
CKPT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('results/checkpoints', exist_ok=True)
if not os.path.exists('results/checkpoints/tiny-imagenet'):
    os.symlink(CKPT_ROOT, 'results/checkpoints/tiny-imagenet')
for sub in ['base', 'at', 'atkd']:
    p = os.path.join('results/checkpoints/tiny-imagenet', sub)
    print(sub, '->', sorted(os.listdir(p))[:3], '...', len(os.listdir(p)), 'files')
assert os.path.isfile('results/checkpoints/tiny-imagenet/base/deit_small_ft.pt'), 'fine-tuned base checkpoint missing'

In [ ]:
# 4. Build /kaggle/working/tiny_if (ImageFolder layout) from albert1994/tinyimagenet
SRC = '/kaggle/input/datasets/albert1994/tinyimagenet/tiny-imagenet-200'
if not os.path.isdir(SRC):
    cands = [p for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True)
             if os.path.isfile(os.path.join(p, 'val', 'val_annotations.txt'))]
    assert cands, 'TinyImageNet not found: attach albert1994/tinyimagenet'
    SRC = cands[0]
print('Source:', SRC)
if not os.path.isdir('/kaggle/working/tiny_if/val_if'):
    sh('prepare_tiny', [sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', SRC, '--out', '/kaggle/working/tiny_if'])
for d in ['val_if', 'train_if']:
    p = os.path.join('/kaggle/working/tiny_if', d)
    print(d, '->', len(os.listdir(p)) if os.path.isdir(p) else 'MISSING', 'class folders')
assert os.path.isdir('/kaggle/working/tiny_if/val_if'), 'val_if was not built: see output above'

In [ ]:
# 5. Environment report (versions, GPU, paths) -> W1/env_report.json
sh('env_report', [sys.executable, 'scripts/env_report.py', '--json', f'{W1}/env_report.json',
                  '--paths', '/kaggle/working/tiny_if/val_if', '/kaggle/working/tiny_if/train_if', CKPT_ROOT])

In [ ]:
# 6. Unit tests. test_bias_guard needs CUDA + bitsandbytes; the others are CPU.
sh('pytest_cpu', [sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/test_runlog.py', 'tests/test_fgsm_rs.py', 'tests/test_tensor_cache.py'])
sh('pytest_bias_guard', [sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/test_bias_guard.py'])

In [ ]:
# 7. Smoke test, one batch per precision (the INT8 attack path is the A-13 risk). Failures are logged, not fatal.
sh('smoke_tiny', [sys.executable, 'run.py', '--exp', 'configs/exp/smoke_tiny.yaml', '--out-root', OUT_ROOT])
for f in sorted(glob.glob(f'{OUT_ROOT}/tiny-imagenet/runs/*.traceback.txt')):
    print('=' * 30, f); print(open(f).read()[-8000:])

In [ ]:
%%writefile /kaggle/working/w1/int8_reload_probe.py
# A-13 hypothesis 1: does load_state_dict of a July INT8 AT checkpoint into a freshly loaded
# (dummy-forwarded) bnb 0.50.2 Linear8bitLt model raise? Read-only on the checkpoint.
import json, sys, traceback
from collections import Counter
sys.path.insert(0, '.')
import torch
from models.loader import load_config, load_model

P = 'results/checkpoints/tiny-imagenet/at/at_int8_epoch01.pt'
out = {'checkpoint': P}
try:
    sd = torch.load(P, map_location='cpu')
    out['weights_only_default_ok'] = True
except Exception as exc:
    out['weights_only_default_ok'] = repr(exc)[:500]
    sd = torch.load(P, map_location='cpu', weights_only=False)
kinds = Counter()
wf_values = Counter()
for k, v in sd.items():
    tail = k.rsplit('.', 1)[-1]
    if tail in ('SCB', 'weight_format', 'CB'):
        kinds[tail] += 1
        if tail == 'weight_format':
            wf_values[repr(v.tolist() if torch.is_tensor(v) else v)] += 1
out['saved_special_keys'] = dict(kinds)
out['saved_weight_format_values'] = dict(wf_values)
out['saved_weight_dtypes'] = dict(Counter(str(v.dtype) for k, v in sd.items() if torch.is_tensor(v) and k.endswith('.weight')))

cfg = load_config('configs/base.yaml')
model = load_model('deit_small', 'int8', cfg, device='cuda')
first = next(m for m in model.modules() if type(m).__name__ == 'Linear8bitLt')
out['fresh_first_layer'] = {'weight_dtype': str(first.weight.dtype),
                            'SCB_is_None': getattr(first.weight, 'SCB', 'n/a') is None,
                            'state_SCB_is_None': getattr(getattr(first, 'state', None), 'SCB', 'n/a') is None}
try:
    res = model.load_state_dict({k: (v.cuda() if torch.is_tensor(v) else v) for k, v in sd.items()}, strict=True)
    out['load_state_dict'] = 'OK'
    out['missing'] = list(res.missing_keys)[:20]
    out['unexpected'] = list(res.unexpected_keys)[:20]
except Exception:
    out['load_state_dict'] = 'RAISED'
    out['traceback'] = traceback.format_exc()[-6000:]
print(json.dumps({k: v for k, v in out.items() if k != 'traceback'}, indent=1))
if 'traceback' in out:
    print(out['traceback'])
json.dump(out, open('/kaggle/working/w1/int8_reload_probe.json', 'w'), indent=1)

In [ ]:
# 8. Run the INT8 reload probe (~1 min)
sh('int8_reload_probe', [sys.executable, f'{W1}/int8_reload_probe.py'])

In [ ]:
%%writefile /kaggle/working/w1/int4_pickle_probe.py
# A-13 addition: did the July INT4 models compute in fp32 (bnb <=0.43 set_compute_type) or fp16?
# CPU only; reads attributes of the pickled Linear4bit modules. Nothing is run through the model.
import json, sys
from collections import Counter
sys.path.insert(0, '.')
import torch

P = 'results/checkpoints/tiny-imagenet/at/at_int4_epoch07_full_model.pt'
m = torch.load(P, map_location='cpu', weights_only=False)
rows = Counter()
n = 0
for name, mod in m.named_modules():
    if type(mod).__name__ != 'Linear4bit':
        continue
    n += 1
    bias = getattr(mod, 'bias', None)
    qs = getattr(mod.weight, 'quant_state', None)
    rows[json.dumps({
        'compute_dtype': str(getattr(mod, 'compute_dtype', 'MISSING')),
        'compute_type_is_set': repr(getattr(mod, 'compute_type_is_set', 'MISSING')),
        'quant_type': str(getattr(mod.weight, 'quant_type', 'MISSING')),
        'bias_dtype': str(bias.dtype) if bias is not None else None,
        'quant_state_dtype': str(getattr(qs, 'dtype', 'MISSING')) if qs is not None else None,
    }, sort_keys=True)] += 1
out = {'checkpoint': P, 'n_linear4bit': n, 'attribute_combinations': {k: v for k, v in rows.items()},
       'top_type': type(m).__name__}
print(json.dumps(out, indent=1))
json.dump(out, open('/kaggle/working/w1/int4_pickle_probe.json', 'w'), indent=1)

In [ ]:
# 9. Run the INT4 pickle probe on CPU (GPU hidden)
sh('int4_pickle_probe', [sys.executable, f'{W1}/int4_pickle_probe.py'], env={'CUDA_VISIBLE_DEVICES': ''})

In [ ]:
# 10. Calibrate (timing + TF32 on/off agreement). ~3-5 min GPU (estimate). Off unless RUN_CALIBRATE = True.
if RUN_CALIBRATE:
    sh('calibrate_tiny', [sys.executable, 'run.py', '--exp', 'configs/exp/calibrate_tiny.yaml', '--out-root', OUT_ROOT])
else:
    print('skipped (RUN_CALIBRATE = False)')

In [ ]:
# 11. Repro check vs July phase1 rows (1,000 images, clean/FGSM/PGD-20, fp32/int8/int4). ~8-12 min GPU (estimate).
# Off unless RUN_REPRO = True. Resumable: rerunning skips finished (precision, attack) results.
if RUN_REPRO:
    sh('repro_check_tiny', [sys.executable, 'run.py', '--exp', 'configs/exp/repro_check_tiny.yaml', '--out-root', OUT_ROOT])
else:
    print('skipped (RUN_REPRO = False)')

In [ ]:
# 12. SUMMARY: copy everything this cell prints back to Claude (also saved to /kaggle/working/w1/summary.txt)
import json, io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print('== wall times / exit codes ==')
    for k, v in TIMES.items():
        print(f'{k:22s} exit={v["exit"]}  {v["seconds"]}s')
    for name in ['pytest_cpu', 'pytest_bias_guard']:
        p = f'{W1}/{name}.log'
        if os.path.exists(p):
            print(f'== {name} (last 25 lines) =='); print(''.join(open(p).readlines()[-25:]))
    p = f'{W1}/env_report.json'
    if os.path.exists(p):
        print('== env_report.json =='); print(open(p).read()[:6000])
    for f in sorted(glob.glob(f'{OUT_ROOT}/tiny-imagenet/runs/*.json')):
        d = json.load(open(f))
        d.pop('per_image', None)
        print('== run', os.path.basename(f), '==')
        print(json.dumps(d, indent=1)[:7000])
    for f in sorted(glob.glob(f'{OUT_ROOT}/tiny-imagenet/runs/*.traceback.txt')):
        print('== traceback', os.path.basename(f), '=='); print(open(f).read()[-5000:])
    for name in ['int8_reload_probe', 'int4_pickle_probe']:
        p = f'{W1}/{name}.json'
        print(f'== {name} ==')
        print(open(p).read()[:7000] if os.path.exists(p) else 'no JSON: see ' + f'{W1}/{name}.log')
text = buf.getvalue()
open(f'{W1}/summary.txt', 'w').write(text)
print(text)